In [122]:
import mikeio
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon

# Load the DFSU file and extract the desired item
dfsu_file = r"D:\Phd Research\Prototype HD Model\Simulation\test_run_Hurricane_Harvey.m21fm - Result Files\area v3_Stat_Total_water_depth.dfsu"
ds = mikeio.open(dfsu_file)
data = ds.read(items="Statistical maximum : Total water depth")

# Load the shapefile for the land boundary
shapefile_path = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"
land_boundary = gpd.read_file(shapefile_path)

# Extract geometry information
geometry = ds.geometry
element_table = geometry.element_table
node_coordinates = geometry.node_coordinates  # This gets the node coordinates

# Get water depth data
water_depth = data[0].to_numpy()
water_depth

# Define the threshold value
threshold = 0.1  # meters

# Identify elements where water depth exceeds the threshold
flooded_elements = np.where(water_depth > threshold)[0]

# Create polygons for flooded elements
flooded_polygons = []
for elem_idx in flooded_elements:
    nodes = element_table[elem_idx] - 1  # zero-index correction
    coords = node_coordinates[nodes]
    flooded_polygons.append(Polygon(coords))
flooded_polygons
# Convert polygons to a GeoDataFrame
flooded_area_gdf = gpd.GeoDataFrame(
    {'geometry': flooded_polygons},
    crs="EPSG:32614"  # UTM Zone 14N
)
output_file = "D:\dipen\output.shp"
flooded_area_gdf.to_file(output_file)

# # polygon = land_boundary.geometry.iloc[0]
# # flooded_area_gdf['geometry'] = flooded_area_gdf.intersection(polygon)

# # Clip the flooded areas with the land boundary
# flooded_area_gdf = gpd.overlay(flooded_area_gdf, land_boundary, how='intersection')


# # Calculate the total flooded area (in square meters)
# total_flooded_area = flooded_area_gdf.area

# # Convert to square kilometers (if needed)
# total_flooded_area_km2 = total_flooded_area / 1e6

# print(f"Total flooded area: {total_flooded_area_km2:.2f} square kilometers")
